# Model Integration: Direct Imports vs. Unified Interface

This notebook demonstrates the two ways to integrate and initialize Chat Models in LangChain:
1. **Approach 1: Direct Imports** (using specific provider packages like `langchain_openai`, `langchain_google_genai`, `langchain_groq`)
2. **Approach 2: Unified Interface (`init_chat_model`)** (a single function to load any provider dynamically via string names, supporting both separate parameters or a shorthand 'provider:model' format)

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from your .env file
load_dotenv(dotenv_path="../.env") #for parent folder
load_dotenv()

# Set keys in the environment for standard LangChain integrations
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


#               or

# for key in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY", "QWEN_API_KEY"]:
#     val = os.getenv(key)
#     if val is not None:
#         os.environ[key] = val


## Approach 1: Direct Imports (Specific Provider Classes)

This approach imports the specific class designed for each provider.
* **Best for:** Explicit code, autocomplete, and setting provider-specific parameters.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 1. Initialize OpenAI (automatically reads OPENAI_API_KEY)
openai_direct = ChatOpenAI(model="gpt-4.1-nano")

# 2. Initialize Qwen (uses ChatOpenAI but overrides the base URL and API Key)
qwen_direct = ChatOpenAI(
    api_key=os.getenv("QWEN_API_KEY"),
    base_url=os.getenv("QWEN_BASE_URL"),
    model=os.getenv("QWEN_MODEL", "qwen-turbo")
)

# 3. Initialize Gemini (automatically reads GOOGLE_API_KEY)
gemini_direct = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 4. Initialize Groq (automatically reads GROQ_API_KEY)
groq_direct = ChatGroq(model="llama-3.3-70b-versatile")

## Approach 2: Unified Interface (init_chat_model)

This approach uses a single `init_chat_model` function. You specify the model name and the provider, and LangChain handles the imports and class instantiation automatically.
* **Best for:** Swapping models dynamically (e.g., in a dropdown or config file) without changing your import code.

In [ ]:
from langchain.chat_models import init_chat_model

# 1. Initialize OpenAI using the unified function
openai_unified = init_chat_model("gpt-4.1-nano", model_provider="openai")

# 2. Initialize Qwen (uses OpenAI-compatible provider, passing specific configuration parameters)
qwen_unified = init_chat_model(
    model=os.getenv("QWEN_MODEL", "qwen-turbo"),
    model_provider="openai",
    api_key=os.getenv("QWEN_API_KEY"),
    base_url=os.getenv("QWEN_BASE_URL")
)

# 3. Initialize Gemini using the unified function
gemini_unified = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

# 4. Initialize Groq using the unified function
groq_unified = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

# 5. Shorthand notation (provider:model) - no model_provider argument is needed
# openai_shorthand = init_chat_model("openai:gpt-4.1-nano")
# gemini_shorthand = init_chat_model("google_genai:gemini-2.5-flash")
# groq_shorthand = init_chat_model("groq:llama-3.3-70b-versatile")


## Testing and Comparing both Approaches

In [4]:
prompt = "Say hello in one word."

print("=== APPROACH 1 (Direct Imports) ===")
print("OpenAI:", openai_direct.invoke(prompt).content)
print("Qwen:", qwen_direct.invoke(prompt).content)
print("Gemini:", gemini_direct.invoke(prompt).content)
print("Groq:", groq_direct.invoke(prompt).content)

print("\n=== APPROACH 2 (init_chat_model) ===")
print("OpenAI:", openai_unified.invoke(prompt).content)
print("Qwen:", qwen_unified.invoke(prompt).content)
print("Gemini:", gemini_unified.invoke(prompt).content)
print("Groq:", groq_unified.invoke(prompt).content)

=== APPROACH 1 (Direct Imports) ===
OpenAI: Hello
Qwen: Hi
Gemini: Hello
Groq: Hello.

=== APPROACH 2 (init_chat_model) ===
OpenAI: Hi
Qwen: Hi
Gemini: Hello
Groq: Hello.


## Streaming and Batching

### 1. Streaming
Most models can stream their output content while it is being generated. By displaying output progressively, streaming significantly improves user experience, particularly for longer responses.

Calling `stream()` returns an iterator that yields output chunks as they are produced. You can use a loop to process each chunk in real-time:

In [ ]:
# We will use the unified Qwen model to demonstrate streaming
model = qwen_unified

# Standard invoke first (waits for the complete response before displaying)
response = model.invoke("Write me a 200 words paragraph on Artificial Intelligence")
print("=== Standard Invoke Response ===")
print(response.content)

=== Standard Invoke Response ===
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks typically requiring human cognition. AI systems can process vast amounts of data, recognize patterns, make decisions, and even improve their performance over time through machine learning. This technology has revolutionized various industries, including healthcare, finance, transportation, and education, by enhancing efficiency, accuracy, and innovation. From virtual assistants and recommendation systems to autonomous vehicles and medical diagnostics, AI is reshaping the way we live and work. However, the rapid advancement of AI also raises ethical concerns, such as job displacement, privacy issues, and biases in decision-making algorithms. As AI continues to evolve, it is crucial to establish responsible guidelines and regulations to ensure its benefits are maximized while minimizing potential risks. The future 

In [18]:
# Streaming the response chunk-by-chunk in real-time
print("=== Streaming Response ===")
for chunk in model.stream("Write me a 200 words paragraph on Artificial Intelligence"):
    print(chunk.content, end="", flush=True)

=== Streaming Response ===
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks typically requiring human cognition. AI systems can process vast amounts of data, recognize patterns, make decisions, and even improve their performance over time through machine learning. This technology has revolutionized various industries, including healthcare, finance, transportation, and education, by enhancing efficiency, accuracy, and innovation. From virtual assistants and recommendation systems to autonomous vehicles and medical diagnostics, AI is reshaping the way we live and work. However, the rapid advancement of AI also raises ethical concerns, such as job displacement, privacy issues, and biases in decision-making algorithms. As AI continues to evolve, it is crucial to establish responsible guidelines and regulations to ensure its benefits are maximized while minimizing potential risks. The future of AI 

### 2. Batching
Batching a collection of independent requests to a model can significantly improve performance and reduce costs, as the processing can be done in parallel:

In [ ]:
# Batching multiple queries in parallel
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])

for i, resp in enumerate(responses):
    print(f"=== Response {i+1} ===")
    print(resp.content[:300] + "...\n")

=== Response 1 ===
Parrots are renowned for their dazzling array of colors, and it's not just for our aesthetic pleasure! Their vibrant plumage serves several crucial evolutionary and biological purposes:

1.  **Sexual Selection and Mate Attraction:** This is arguably the most significant reason.
    *   **Signaling H...

=== Response 2 ===
Airplanes fly by expertly manipulating four fundamental forces: **Lift, Weight, Thrust, and Drag.** When these forces are balanced in a specific way, an aircraft can take off, fly, and land.

Let's break down each force and how they work together:

1.  **Lift (The upward force):**
    *   **How it's...

=== Response 3 ===
Quantum computing is a **new type of computing** that harnesses the principles of **quantum mechanics** to solve complex problems that are intractable (too difficult or time-consuming) for even the most powerful classical supercomputers.

Instead of relying on the classical bits that store informati...



In [20]:
# Batching with concurrency configuration to limit parallel requests
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
], 

config={
    'max_concurrency': 2,  # Limit to 2 parallel calls
})

for i, resp in enumerate(responses):
    print(f"=== Response {i+1} ===")
    print(resp.content[:150] + "...\n")

=== Response 1 ===
Parrots have colorful feathers for several important reasons, primarily related to **survival, communication, and reproduction**. Here's a breakdown o...

=== Response 2 ===
Airplanes fly due to a combination of four fundamental forces: **lift**, **gravity (weight)**, **thrust**, and **drag**. These forces interact in a wa...

=== Response 3 ===
**Quantum computing** is a type of computing that uses the principles of **quantum mechanics** to perform operations on data. Unlike classical compute...

